# 01b — ViTPose++ Feature Extraction (Colab A100 edition)

Run `00_manifest_qc.ipynb` trước hoặc dùng manifest từ Kaggle dataset
`vnhtbo/manifest`. Writes resumable ViTPose++ pose artifacts.

**Thay đổi so với phiên bản Kaggle T4×2:**

- Bootstrap qua `aic_colab_utils.setup_aic2026_environment()`.
- Batch detection (RTDetr) thay vì 1 ảnh/iter.
- Batch pose estimation: stack person crops cùng batch để qua VitPose++.
- BF16 autocast cho cả detector + pose.
- DataLoader 8 workers + pin_memory + prefetch.
- Async sync mỗi chunk NPZ sang Drive.
- Resume-safe: skip chunks đã có trên local hoặc Drive.


In [ ]:
# === Bootstrap & config ===
from pathlib import Path
import gc
import os
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    mirror_dataset_to_drive,
    mirror_raw_as_tar_split,        # NEW: tar+split raw → Drive
    setup_aic2026_environment,
    select_a100_device,
    sync_chunk_to_drive,
    wait_for_pending_syncs,
    chunk_done,
    save_npz_atomic,
    chunk_file,
    maybe_clear_cuda,
)

PATHS = setup_aic2026_environment()
INPUT_ROOT = PATHS['input_root']
MANIFEST_DIR = PATHS['manifest_dir']
OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']
LOCAL_ROOT = PATHS['local_root']
POSE_DIR = OUTPUT_ROOT / 'pose' / 'vitposepp'
POSE_DIR.mkdir(parents=True, exist_ok=True)

# --- Run knobs ---
TRAIN_START_IDX = 0
TRAIN_END_IDX = None

SMOKE_TEST = False
SMOKE_N_TRAIN = 100
SMOKE_N_GALLERY = 100

RUN_TRAIN_POSE = True
RUN_GALLERY_POSE = True

# Drive persistence strategy (xem 01a docstring để biết chi tiết)
MIRROR_RAW_STRATEGY = 'tar'   # 'tar' | 'rsync' | False

# --- A100-80GB tuned hyperparams ---
POSE_MODEL_ID = 'usyd-community/vitpose-plus-huge'
DETECTOR_MODEL_ID = 'PekingU/rtdetr_r50vd_coco_o365'
DETECTION_THRESHOLD = 0.30
KEEP_TOPK_PERSONS = 1

DETECTION_BATCH = 128          # A100-40GB: 32; A100-80GB: 128
POSE_BATCH = 128               # A100-40GB: 32; mỗi image 1 top-1 crop nên có thể batch lớn
POSE_CHUNK_SIZE = 8192         # A100-40GB: 4096
NUM_WORKERS = 12               # high-RAM cho phép 12 workers
PREFETCH_FACTOR = 4
USE_BF16 = True
ASYNC_DRIVE_SYNC = True
POSE_CLEAR_CACHE_EVERY_N_BATCHES = 100  # 80GB ít cần clear hơn

print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('POSE_DIR:', POSE_DIR)
print(f'MIRROR_RAW_STRATEGY = {MIRROR_RAW_STRATEGY!r}')


In [ ]:
# === Install deps ===
def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

try:
    import transformers  # noqa: F401
except Exception:
    pip_install('transformers>=4.52.3', 'accelerate', 'timm', 'einops', 'safetensors')

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
# === Device + manifest load ===
device = select_a100_device()
amp_dtype = torch.bfloat16 if (USE_BF16 and device.type == 'cuda') else torch.float16
detector_device = device
pose_device = device

train_df = pd.read_parquet(MANIFEST_DIR / 'train_manifest.parquet')
gallery_df = pd.read_parquet(MANIFEST_DIR / 'gallery_manifest.parquet')

def remap_path(p):
    if not isinstance(p, str) or not p:
        return p
    if p.startswith(str(INPUT_ROOT)):
        return p
    for anchor in ('annotation/', 'train/imgs_', 'name-masked_test-set/', 'gallery/'):
        i = p.find(anchor)
        if i >= 0:
            return str(INPUT_ROOT / p[i:])
    return p

for df_ in (train_df, gallery_df):
    if 'image_path' in df_.columns:
        df_['image_path'] = df_['image_path'].map(remap_path)

if SMOKE_TEST:
    train_df = train_df.head(SMOKE_N_TRAIN).copy()
    gallery_df = gallery_df.head(SMOKE_N_GALLERY).copy()

train_df = train_df[train_df['image_path'].astype(str).str.len() > 0].reset_index(drop=True)
if TRAIN_END_IDX is not None:
    train_df = train_df.iloc[TRAIN_START_IDX:TRAIN_END_IDX].reset_index(drop=True)
else:
    train_df = train_df.iloc[TRAIN_START_IDX:].reset_index(drop=True)

print('train:', train_df.shape, '| gallery:', gallery_df.shape)


In [ ]:
# === Load detector + pose model ===
from transformers import AutoProcessor, RTDetrForObjectDetection, VitPoseForPoseEstimation

maybe_clear_cuda()
print('Loading detector:', DETECTOR_MODEL_ID)
detector_processor = AutoProcessor.from_pretrained(DETECTOR_MODEL_ID)
detector_model = (
    RTDetrForObjectDetection.from_pretrained(DETECTOR_MODEL_ID)
    .to(detector_device).eval()
)

print('Loading pose model:', POSE_MODEL_ID)
pose_processor = AutoProcessor.from_pretrained(POSE_MODEL_ID)
pose_model = (
    VitPoseForPoseEstimation.from_pretrained(POSE_MODEL_ID)
    .to(pose_device).eval()
)
print('Loaded pose model:', POSE_MODEL_ID)


In [ ]:
# === Batched pose extraction ===
class RawImageDataset(Dataset):
    """Load PIL.Image kèm id/path. Trả về raw RGB image (đã decode)."""

    def __init__(self, df, id_col):
        self.paths = df['image_path'].astype(str).tolist()
        self.ids = df[id_col].astype(str).tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        path = self.paths[i]
        try:
            with Image.open(path) as img:
                arr = np.asarray(img.convert('RGB'))
            return arr, self.ids[i], path, True
        except Exception:
            return np.zeros((1, 1, 3), dtype=np.uint8), self.ids[i], path, False


def _collate_raw(batch):
    return batch  # giữ nguyên list, không stack — image kích thước khác nhau


def _full_image_box(width, height):
    return (
        np.array([[0.0, 0.0, float(width), float(height)]], dtype=np.float32),
        np.array([0.0], dtype=np.float32),
        False,
    )


@torch.inference_mode()
def detect_batch(pil_images):
    """Run RTDetr trên batch PIL. Trả về list (boxes_xywh, scores, detected)."""
    inputs = detector_processor(images=pil_images, return_tensors='pt').to(detector_device)
    with torch.autocast(device_type=detector_device.type, dtype=amp_dtype,
                        enabled=(detector_device.type == 'cuda')):
        outputs = detector_model(**inputs)
    target_sizes = torch.tensor(
        [(im.size[1], im.size[0]) for im in pil_images], device=detector_device,
    )
    results = detector_processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=DETECTION_THRESHOLD,
    )

    out = []
    for img, res in zip(pil_images, results):
        w, h = img.size
        labels = res['labels'].detach().cpu().numpy()
        keep = labels == 0
        if keep.sum() == 0:
            out.append(_full_image_box(w, h))
            continue
        boxes_xyxy = res['boxes'].detach().cpu().numpy()[keep].astype(np.float32)
        scores = res['scores'].detach().cpu().numpy()[keep].astype(np.float32)
        boxes = boxes_xyxy.copy()
        boxes[:, 2] = np.maximum(1.0, boxes_xyxy[:, 2] - boxes_xyxy[:, 0])
        boxes[:, 3] = np.maximum(1.0, boxes_xyxy[:, 3] - boxes_xyxy[:, 1])
        boxes[:, 0] = np.clip(boxes[:, 0], 0, w - 1)
        boxes[:, 1] = np.clip(boxes[:, 1], 0, h - 1)
        boxes[:, 2] = np.clip(boxes[:, 2], 1, w)
        boxes[:, 3] = np.clip(boxes[:, 3], 1, h)
        area = boxes[:, 2] * boxes[:, 3]
        rank = scores * np.sqrt(np.maximum(area, 1.0))
        order = np.argsort(-rank)[:KEEP_TOPK_PERSONS]
        out.append((boxes[order], scores[order], True))
    return out


@torch.inference_mode()
def pose_batch(pil_images, det_results):
    """Run VitPose++ trên batch (image, boxes).

    Trả về list (keypoints[17,3], bbox_norm[4], det_score, valid).
    """
    boxes_list = [d[0] for d in det_results]
    inputs = pose_processor(pil_images, boxes=boxes_list, return_tensors='pt').to(pose_device)
    if 'vitpose-plus' in str(POSE_MODEL_ID):
        dataset_index = torch.zeros(
            inputs['pixel_values'].shape[0], dtype=torch.long, device=pose_device,
        )
        with torch.autocast(device_type=pose_device.type, dtype=amp_dtype,
                            enabled=(pose_device.type == 'cuda')):
            outputs = pose_model(**inputs, dataset_index=dataset_index)
    else:
        with torch.autocast(device_type=pose_device.type, dtype=amp_dtype,
                            enabled=(pose_device.type == 'cuda')):
            outputs = pose_model(**inputs)
    pose_results = pose_processor.post_process_pose_estimation(
        outputs, boxes=boxes_list,
    )

    finals = []
    for img, det, pr in zip(pil_images, det_results, pose_results):
        w, h = img.size
        boxes, det_scores, detected = det
        if not pr:
            finals.append((
                np.zeros((17, 3), dtype=np.float32),
                np.array([0, 0, 1, 1], dtype=np.float32),
                0.0,
                False,
            ))
            continue
        item = pr[0]
        kp = item['keypoints'].detach().cpu().numpy().astype(np.float32)
        sc = item['scores'].detach().cpu().numpy().astype(np.float32)
        kp[:, 0] = kp[:, 0] / max(w, 1)
        kp[:, 1] = kp[:, 1] / max(h, 1)
        kp = np.concatenate([kp, sc[:, None]], axis=1)
        box = boxes[0].astype(np.float32).copy()
        box[0] /= max(w, 1)
        box[1] /= max(h, 1)
        box[2] /= max(w, 1)
        box[3] /= max(h, 1)
        score = float(det_scores[0]) if len(det_scores) else 0.0
        finals.append((kp, box, score, bool(detected)))
    return finals


def extract_pose_df_to_npz(df, id_col, out_path):
    out_path = Path(out_path)
    if chunk_done(out_path, DRIVE_OUTPUT_ROOT, LOCAL_ROOT):
        print('Skip existing', out_path.name)
        return
    if len(df) == 0:
        return
    df = df.reset_index(drop=True)

    loader = DataLoader(
        RawImageDataset(df, id_col),
        batch_size=DETECTION_BATCH,
        num_workers=NUM_WORKERS,
        pin_memory=False,                     # raw numpy, không cần pin
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
        collate_fn=_collate_raw,
        shuffle=False,
        drop_last=False,
    )

    ids, paths, keypoints, bboxes, pose_scores, valid = [], [], [], [], [], []
    pbar = tqdm(total=len(df), desc=f'pose → {out_path.name}', unit='img')
    batch_count = 0

    for raw_batch in loader:
        # raw_batch: list of (np.array RGB, id, path, ok_load)
        pil_images, b_ids, b_paths, ok_loads = [], [], [], []
        for arr, _id, _path, ok in raw_batch:
            pil = Image.fromarray(arr) if ok else Image.new('RGB', (8, 8))
            pil_images.append(pil)
            b_ids.append(_id)
            b_paths.append(_path)
            ok_loads.append(ok)

        # Detection (batch) + pose (sub-batch để pose model không OOM)
        det_results = detect_batch(pil_images)

        # Pose có thể batch lớn hơn detection do mỗi image chỉ 1 crop top-1
        pose_outputs = []
        for sub_start in range(0, len(pil_images), POSE_BATCH):
            sub_imgs = pil_images[sub_start:sub_start + POSE_BATCH]
            sub_dets = det_results[sub_start:sub_start + POSE_BATCH]
            pose_outputs.extend(pose_batch(sub_imgs, sub_dets))

        for _id, _path, ok_load, (kp, bbox_n, det_score, ok_det) in zip(
            b_ids, b_paths, ok_loads, pose_outputs,
        ):
            ids.append(_id)
            paths.append(_path)
            keypoints.append(kp)
            bboxes.append(bbox_n)
            pose_scores.append(det_score)
            valid.append(bool(ok_load and ok_det))

        pbar.update(len(raw_batch))
        batch_count += 1
        if batch_count % POSE_CLEAR_CACHE_EVERY_N_BATCHES == 0:
            maybe_clear_cuda()

    pbar.close()
    keypoints_arr = np.stack(keypoints).astype('float16')
    bboxes_arr = np.stack(bboxes).astype('float16')
    save_npz_atomic(
        out_path,
        ids=np.array(ids),
        paths=np.array(paths),
        keypoints=keypoints_arr,
        bbox=bboxes_arr,
        pose_score=np.array(pose_scores, dtype=np.float16),
        valid=np.array(valid, dtype=bool),
        pose_model=np.array([POSE_MODEL_ID]),
        detector_model=np.array([DETECTOR_MODEL_ID]),
    )
    if ASYNC_DRIVE_SYNC:
        sync_chunk_to_drive(out_path, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
    maybe_clear_cuda()
    print('Wrote', out_path, keypoints_arr.shape, 'valid:', int(np.sum(valid)))


def extract_train_pose_chunked():
    out_dir = POSE_DIR / 'train'
    out_dir.mkdir(parents=True, exist_ok=True)
    for start in range(0, len(train_df), POSE_CHUNK_SIZE):
        end = min(start + POSE_CHUNK_SIZE, len(train_df))
        out_path = chunk_file(out_dir, start, end)
        extract_pose_df_to_npz(train_df.iloc[start:end], 'image_id', out_path)


In [ ]:
# === Run extraction ===
if RUN_TRAIN_POSE:
    extract_train_pose_chunked()

if RUN_GALLERY_POSE:
    extract_pose_df_to_npz(gallery_df, 'gallery_id', POSE_DIR / 'gallery.npz')

wait_for_pending_syncs()

# === Mirror raw + manifests lên Drive (chỉ chạy nếu 01a chưa chạy trước đó) ===
#
# Nếu bạn đã chạy 01a trước 01b, raw đã được mirror lên Drive rồi —
# `mirror_raw_as_tar_split` sẽ skip vì marker `.tar_complete` đã có.
# Khả năng cao bạn chỉ muốn skip toàn bộ phần này khi run 01b sau 01a.
if MIRROR_RAW_STRATEGY == 'tar':
    mirror_raw_as_tar_split(PATHS)                                              # raw → tar parts (skip nếu đã có)
    mirror_dataset_to_drive(PATHS, include_raw=False, include_manifests=True)
elif MIRROR_RAW_STRATEGY == 'rsync':
    mirror_dataset_to_drive(PATHS, include_raw=True, include_manifests=True)
else:
    mirror_dataset_to_drive(PATHS, include_raw=False, include_manifests=True)

print('Done.')
